# SkillCorner Body Pose Viewer · coleta dos clipes

Este é o único pipeline do projeto. Ele baixa os arquivos oficiais de **Body Pose** e **Game Intelligence**, seleciona `player_possession` encerradas em `shot` e exporta replays compactos para o visualizador.

A sincronização segue a documentação oficial: Body Pose opera a 25 FPS, enquanto Dynamic Events opera a 10 FPS; portanto `pose_frame = 2.5 × event_frame`. Nenhuma pose é interpolada ou inferida.

In [ ]:
from __future__ import annotations

import csv
import json
import urllib.request
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
SOURCE = DATA / 'source'
CLIPS = DATA / 'clips'
SOURCE.mkdir(parents=True, exist_ok=True)
CLIPS.mkdir(parents=True, exist_ok=True)

MATCH_IDS = (1925299, 1996435)
POSE_FPS = 25
EVENT_FPS = 10
SAMPLE_EVERY = 1  # 25 FPS nativos para reprodução fluida
SECONDS_BEFORE = 3
SECONDS_AFTER = 2

JOINTS = ['nose','neck','lEye','rEye','lEar','rEar','lShoulder','rShoulder','lElbow','rElbow','lWrist','rWrist','lThumb','rThumb','lPinky','rPinky','midHip','lHip','rHip','lKnee','rKnee','lAnkle','rAnkle','lHeel','rHeel','lBigToe','rBigToe','lSmallToe','rSmallToe']
BONES = [('nose','neck'),('nose','lEye'),('nose','rEye'),('lEye','lEar'),('rEye','rEar'),('neck','lShoulder'),('neck','rShoulder'),('lShoulder','lElbow'),('lElbow','lWrist'),('lWrist','lThumb'),('lWrist','lPinky'),('rShoulder','rElbow'),('rElbow','rWrist'),('rWrist','rThumb'),('rWrist','rPinky'),('neck','midHip'),('midHip','lHip'),('midHip','rHip'),('lHip','lKnee'),('lKnee','lAnkle'),('lAnkle','lHeel'),('lHeel','lBigToe'),('lHeel','lSmallToe'),('rHip','rKnee'),('rKnee','rAnkle'),('rAnkle','rHeel'),('rHeel','rBigToe'),('rHeel','rSmallToe')]
BONE_INDICES = [[JOINTS.index(a), JOINTS.index(b)] for a, b in BONES]

def download(url: str, destination: Path) -> Path:
    if destination.is_file() and destination.stat().st_size:
        return destination
    print(f'Downloading {destination.name} ...')
    urllib.request.urlretrieve(url, destination)
    return destination

def source_files(match_id: int) -> dict[str, Path]:
    github = f'https://raw.githubusercontent.com/SkillCorner/opendata/master/data/matches/{match_id}'
    huggingface = 'https://huggingface.co/datasets/SkillCorner/opendata-bodypose/resolve/main/raw'
    return {
        'pose': download(f'{huggingface}/{match_id}.jsonl.zip', SOURCE / f'{match_id}.jsonl.zip'),
        'events': download(f'{github}/{match_id}_dynamic_events.csv', SOURCE / f'{match_id}_dynamic_events.csv'),
        'match': download(f'{github}/{match_id}_match.json', SOURCE / f'{match_id}_match.json'),
    }

files = {match_id: source_files(match_id) for match_id in MATCH_IDS}


In [ ]:
def number(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def load_shots(path: Path, match_id: int) -> list[dict]:
    with path.open(encoding='utf-8-sig', newline='') as handle:
        rows = [row for row in csv.DictReader(handle) if row['event_type'] == 'player_possession']
    score_seen = defaultdict(int)
    shots = []
    for row in rows:
        team = row['team_shortname']
        score = int(number(row['team_score']))
        score_jump = score > score_seen[team]
        if row['end_type'] == 'shot':
            shot_frame = round(int(row['frame_end']) * POSE_FPS / EVENT_FPS)
            start = max(0, round(int(row['frame_start']) * POSE_FPS / EVENT_FPS) - SECONDS_BEFORE * POSE_FPS)
            end = shot_frame + SECONDS_AFTER * POSE_FPS
            clip_id = f"{match_id}_{row['event_id'].replace('_', '-')}"
            shots.append({
                'id': clip_id, 'match_id': match_id, 'event_id': row['event_id'],
                'file': f'data/clips/{clip_id}.json', 'period': int(row['period']),
                'time': row['time_end'], 'player_id': int(row['player_id']),
                'player': row['player_name'], 'team': team,
                'header': row['is_header'] == 'True',
                'goal': row['lead_to_goal'] == 'True' or score_jump,
                'x': round(number(row['x_end'], number(row['x_start'])), 2),
                'y': round(number(row['y_end'], number(row['y_start'])), 2),
                'attacking_side': row['attacking_side'],
                'pose_frame': shot_frame, 'start_frame': start, 'end_frame': end,
            })
        score_seen[team] = max(score_seen[team], score)
    return shots

def compact_frame(frame: dict, player_teams: dict[int, int], active_ranges: dict[int, tuple[float, float]]) -> list:
    players = []
    for player in frame.get('player_data') or []:
        player_id = int(player['player_id'])
        tracking_frame = int(frame['frame']) * EVENT_FPS / POSE_FPS
        active_range = active_ranges.get(player_id)
        if active_range is None or not active_range[0] <= tracking_frame <= active_range[1]:
            continue
        joints = player.get('joints')
        coordinates, errors = None, []
        if isinstance(joints, dict):
            coordinates = []
            for name in JOINTS:
                joint = joints.get(name) or {}
                xyz = joint.get('xyz')
                if not isinstance(xyz, list) or len(xyz) < 3:
                    coordinates.extend((None, None, None))
                else:
                    coordinates.extend(round(number(value), 3) for value in xyz[:3])
                    if joint.get('p90_mae_cm') is not None:
                        errors.append(number(joint['p90_mae_cm']))
        players.append([
            player_id, player_teams.get(player_id, 2),
            round(number(player.get('x')), 2), round(number(player.get('y')), 2),
            round(sum(errors) / len(errors), 1) if errors else None, coordinates,
        ])
    ball = frame.get('ball_data') or {}
    compact_ball = None if ball.get('x') is None else [round(number(ball.get(k)), 3) for k in ('x', 'y', 'z')]
    return [int(frame['frame']), frame.get('timestamp'), compact_ball, players]

all_clips, match_entries, player_entries = [], [], {}
for match_id in MATCH_IDS:
    match = json.loads(files[match_id]['match'].read_text(encoding='utf-8'))
    home_id, away_id = match['home_team']['id'], match['away_team']['id']
    player_teams = {int(p['id']): 0 if p['team_id'] == home_id else 1 if p['team_id'] == away_id else 2 for p in match['players']}
    active_ranges = {}
    for player in match['players']:
        total = (player.get('playing_time') or {}).get('total')
        if total:
            active_ranges[int(player['id'])] = (
                total.get('start_frame') or 0,
                total.get('end_frame') if total.get('end_frame') is not None else float('inf'),
            )
    for player in match['players']:
        player_entries[str(player['id'])] = {
            'name': player['short_name'], 'number': player.get('number'),
            'team': player_teams[int(player['id'])],
        }
    shots = load_shots(files[match_id]['events'], match_id)
    frames_by_clip = {shot['id']: [] for shot in shots}
    windows = sorted(shots, key=lambda shot: shot['start_frame'])
    next_window, active = 0, []
    maximum_frame = max(shot['end_frame'] for shot in shots)
    with zipfile.ZipFile(files[match_id]['pose']) as archive:
        member = f'{match_id}.jsonl'
        with archive.open(member) as stream:
            for line in stream:
                frame = json.loads(line)
                frame_number = int(frame['frame'])
                while next_window < len(windows) and windows[next_window]['start_frame'] <= frame_number:
                    active.append(windows[next_window])
                    next_window += 1
                active = [shot for shot in active if shot['end_frame'] >= frame_number]
                if active and frame_number % SAMPLE_EVERY == 0:
                    compact = compact_frame(frame, player_teams, active_ranges)
                    for shot in active:
                        frames_by_clip[shot['id']].append(compact)
                if frame_number > maximum_frame and next_window == len(windows):
                    break
    for shot in shots:
        frames = frames_by_clip[shot['id']]
        payload = {'fps': POSE_FPS / SAMPLE_EVERY, 'shot_frame': shot['pose_frame'], 'frames': frames}
        destination = ROOT / shot['file']
        destination.write_text(json.dumps(payload, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')
        shot['frames'] = len(frames)
        shot['pose_coverage'] = round(sum(any(player[5] for player in frame[3]) for frame in frames) / max(1, len(frames)), 3)
        shot['mean_tracked_players'] = round(sum(len(frame[3]) for frame in frames) / max(1, len(frames)), 1)
        all_clips.append({k: v for k, v in shot.items() if k not in ('start_frame', 'end_frame')})
    match_entries.append({
        'id': match_id, 'date': match['date_time'][:10],
        'home': match['home_team']['short_name'], 'away': match['away_team']['short_name'],
        'home_score': match['home_team_score'], 'away_score': match['away_team_score'],
        'shots': len(shots), 'goals': sum(shot['goal'] for shot in shots),
    })
    print(f'{match_id}: {len(shots)} finalizações exportadas')

manifest = {
    'name': 'SkillCorner Body Pose Viewer',
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source': 'SkillCorner Open Data · Body Pose + Game Intelligence',
    'pose_fps': POSE_FPS, 'playback_fps': POSE_FPS / SAMPLE_EVERY,
    'joints': JOINTS, 'bones': BONE_INDICES, 'players': player_entries,
    'matches': match_entries, 'clips': sorted(all_clips, key=lambda clip: (clip['match_id'], clip['pose_frame'])),
}
(DATA / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"{len(all_clips)} clipes prontos em {CLIPS}")
